# 0. Import Libraries

In [ ]:
import os, sys
from pathlib import Path

def add_project_path(module_folder="model_implementation"):
    candidates = [
        os.path.abspath("."),
        os.path.abspath("../src"),
        os.path.abspath("src"),
    ]

    for path in candidates:
        if os.path.exists(os.path.join(path, module_folder)):
            if path not in sys.path:
                sys.path.append(path)
            return path

    raise ImportError(f"Could not find '{module_folder}' in current or parent directory")

SRC_PATH = add_project_path("model_implementation")
add_project_path("cnn")
add_project_path("rnn")
ROOT = Path(SRC_PATH).parent.resolve()
print("ROOT:", ROOT)

In [ ]:
# Jalankan cell ini hanya bila environment belum memiliki dependency.
# import subprocess
# subprocess.check_call([sys.executable, "-m", "pip", "install", "tensorflow", "scikit-learn", "pandas", "matplotlib", "nltk", "pillow"])

In [ ]:
from pathlib import Path
import os

import matplotlib.pyplot as plt
import numpy as np
tf = None

from common.io import load_npy
from rnn.preprocess import prep_data, load_vocab

# 1. Global Variables

In [ ]:
SEED = 42
IMAGE_SIZE = (150, 150)
BATCH_SIZE = 32
EPOCHS = 10
MAX_CAPTION_LENGTH = 34

np.random.seed(SEED)
plt.style.use("seaborn-v0_8")

DATA_DIR = ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
FEATURE_DIR = DATA_DIR / "features"
VOCAB_DIR = DATA_DIR / "vocab"
MODEL_DIR = ROOT / "models"
CNN_MODEL_DIR = MODEL_DIR / "cnn"
RNN_MODEL_DIR = MODEL_DIR / "rnn"
REPORT_DIR = ROOT / "reports"
TABLE_DIR = REPORT_DIR / "tables"
FIG_DIR = REPORT_DIR / "figures"

for path in [FEATURE_DIR, VOCAB_DIR, CNN_MODEL_DIR, RNN_MODEL_DIR, TABLE_DIR, FIG_DIR]:
    path.mkdir(parents=True, exist_ok=True)

CATEGORIES = {
    "buildings": 0,
    "forest": 1,
    "glacier": 2,
    "mountain": 3,
    "sea": 4,
    "street": 5,
}
INV_CATEGORIES = {v: k for k, v in CATEGORIES.items()}

if tf is not None:
    gpu_devices = tf.config.list_physical_devices("GPU")
    if gpu_devices:
        print("GPU Detected:", gpu_devices)
        for gpu in gpu_devices:
            tf.config.experimental.set_memory_growth(gpu, True)
    else:
        print("No GPU found, defaulting to CPU.")

# 2. Caption Preprocessing

In [ ]:
def read_flickr8k_caption_pairs(path):
    pairs = []
    with Path(path).open('r', encoding='utf-8') as file:
        for line_number, line in enumerate(file):
            line = line.strip()
            if not line:
                continue
            if line_number == 0 and line.lower().startswith('image,'):
                continue
            if ',' in line:
                image_name, caption = line.split(',', 1)
            else:
                image_name, caption = line.split(None, 1)
            image_id = Path(image_name.split('#')[0]).stem
            pairs.append((image_id, caption))
    return pairs

CAPTION_FILE = RAW_DIR / "flickr8k" / "captions.txt"
CAPTION_PATH = VOCAB_DIR / "caption_sequences.npy"
VOCAB_PATH = VOCAB_DIR / "vocab.json"
CAPTION_ID_PATH = VOCAB_DIR / "caption_image_ids.txt"

caption_sequences = None
caption_image_ids = []
word_to_index = index_to_word = None

if CAPTION_FILE.exists():
    caption_pairs = read_flickr8k_caption_pairs(CAPTION_FILE)
    caption_image_ids = [image_id for image_id, _ in caption_pairs]
    captions = [caption for _, caption in caption_pairs]
    caption_sequences, word_to_index, index_to_word = prep_data(captions, max_length=MAX_CAPTION_LENGTH, out_dir=VOCAB_DIR)
    CAPTION_ID_PATH.write_text('\n'.join(caption_image_ids), encoding='utf-8')
    print("caption pairs:", len(caption_pairs))
    print("unique images:", len(set(caption_image_ids)))
    print("sequences:", caption_sequences.shape)
    print("vocab:", len(word_to_index))
elif CAPTION_PATH.exists() and VOCAB_PATH.exists() and CAPTION_ID_PATH.exists():
    caption_sequences = load_npy(CAPTION_PATH).astype('int32')
    caption_image_ids = [line.strip() for line in CAPTION_ID_PATH.read_text(encoding='utf-8').splitlines() if line.strip()]
    word_to_index, index_to_word = load_vocab(VOCAB_DIR)
    print("Loaded caption artifacts:", caption_sequences.shape, len(caption_image_ids), len(word_to_index))
else:
    print("Caption artifacts belum tersedia.")